# Đo hybrid BM25 + dense — nâng trần pool hay không?

EDA §15.7: reranker đã **bão hoà pool** (pool 20, k=10 ăn 100% khoảng trống). Năng lực
xếp hạng hết là chỗ nghẽn. Đường duy nhất còn lại của dense là kéo gold từ hạng 51+ vào
top-50 — nâng **trần**.

Notebook này KHÔNG chạy cross-encoder: chỉ đo trần, nhanh hơn ~50 lần. Reranker lo phần
sau với hiệu suất đã biết (ăn ~72% khoảng trống → +1 điểm trần ≈ +0,7 điểm R@5).

**Ngưỡng chốt trước khi chạy** — mốc phải vượt là BM25 pool 100 (trần 97,3%, giá 0 params):

| trần hybrid @50 | kết luận |
|---|---|
| < 96,2% | dense kéo tụt → vứt |
| 96,2–97,3% | không hơn pool 100 vốn miễn phí params → vứt |
| ≥ 98% | +1,3 điểm R@5, đáng 135M params → gắn |

Cần gắn: `official` · `legalqa-v10-src` · `legalqa-warmup` · GPU T4 x2 · Internet ON

## 1 — Nạp codebase từ Dataset

Codebase vào bằng **Kaggle Dataset** `legalqa-v10-src`, không clone GitHub: repo private,
fine-grained PAT qua Kaggle Secrets tốn hai vòng debug mà vẫn 404.

Cần gắn: **Input → Add Input → Datasets → `legalqa-v10-src`**

Dòng `codebase:` in ra commit của bản đang chạy. Đối chiếu với `git log -1` trên máy dev —
lệch nghĩa là dataset cũ, chạy `./scripts/push_dataset.sh` rồi *Add Input* lại bản mới.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Không clone GitHub nữa: repo private, fine-grained PAT + Kaggle Secrets tốn hai vòng
# debug mà vẫn 404. Codebase đi vào bằng Kaggle Dataset — không auth, chạy được cả khi
# Internet OFF. Đẩy bản mới bằng scripts/push_dataset.sh trên máy dev.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    # Kaggle đôi khi giữ nguyên .zip thay vì bung; tự bung vào working.
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

# Copy sang working: scripts ghi bm25.pkl/chunks.jsonl, /kaggle/input là read-only.
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")
print("số GPU  :", torch.cuda.device_count())

# P100 (sm_60) không chạy được fp16 với torch bản trên Kaggle. Code tự tụt về
# fp32 nên không vỡ, nhưng chậm ~2x — báo to để biết mà đổi Accelerator.
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy -> fp32, đổi Accelerator sang T4 x2)")


## 2 — Tìm dữ liệu thi

Dò thay vì bắt gõ tay đường dẫn: tên dataset khác nhau tuỳ bạn gắn competition hay upload zip.
Mốc nhận dạng là thư mục `selected-contexts/` — thứ duy nhất chắc chắn có.

In [ ]:
from pathlib import Path

cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

train = DATA_DIR / "train.json"
assert train.exists(), f"Thiếu train.json cạnh selected-contexts/ trong {DATA_DIR}"

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["TRAIN_JSON"] = str(train)
os.environ["CLEAN_DIR"] = "/kaggle/working/clean"
os.environ["PYTHONPATH"] = "src"

n = len(list((DATA_DIR / "selected-contexts").glob("*.json")))
print(f"DATA_DIR  {DATA_DIR}")
print(f"corpus    {n:,} file  (máy dev: 8.532)")

## 3 — clean → chunk → gold (CPU, ~12 phút)

Ba số phải khớp máy dev, lệch là dữ liệu khác bản — mọi số sau đó vô nghĩa:
`Document 8.512` · `Chunk 287.358` · `gold 6.250/7.000 = 89,3%`

In [ ]:
!mkdir -p $CLEAN_DIR
!python scripts/clean_data.py
!python scripts/build_chunks.py
!python scripts/build_gold.py
# warmup.json: 500 câu CÓ Reference answer và KHÔNG nằm trong train. Mọi ngưỡng pool/K
# đang chốt trên train -> cần tập độc lập kiểm lại, nếu không là chốt tham số trên chính
# tập đã dùng để quan sát. Máy dev dựng được 456/500 = 91,2%.
#
# Nó KHÔNG nằm cạnh train.json (dataset `official` chỉ có official/) -> dò cả /kaggle/input.
from pathlib import Path
warm = next(Path("/kaggle/input").glob("**/warmup.json"), None)
if warm:
    !TRAIN_JSON=$warm GOLD_OUT=gold_warmup.jsonl python scripts/build_gold.py
    os.environ["CO_WARMUP"] = "1"
else:
    os.environ["CO_WARMUP"] = ""
    print("KHÔNG thấy warmup.json — gắn dataset legalqa-warmup để có tập độc lập")


## 4 — Chọn bi-encoder

Ngân sách còn **340M** params (4B − Qwen 3,09B − reranker 0,568B). Loại dùng được:

| model | params | ghi chú |
|---|---:|---|
| `intfloat/multilingual-e5-small` | ~118M | 384 chiều, cần prefix `query:`/`passage:` |
| `bkai-foundation-models/vietnamese-bi-encoder` | ~135M | PhoBERT-base, train cho tiếng Việt |
| ~~`AITeamVN/Vietnamese_Embedding`~~ | 568M | **vượt trần**, XLM-R large |
| ~~`BAAI/bge-m3`~~ | 568M | **vượt trần** |

Đo cả hai model vừa ngân sách rồi so — không đoán cái nào hợp văn bản quy phạm.

In [ ]:
import json
from huggingface_hub import snapshot_download
from transformers import AutoConfig

MODELS = ["intfloat/multilingual-e5-small", "bkai-foundation-models/vietnamese-bi-encoder"]
PATHS = {}
for m in MODELS:
    PATHS[m] = snapshot_download(m)
    cfg = AutoConfig.from_pretrained(PATHS[m])
    # Ước params: 12 lớp × (12 h² attention+FFN) + vocab × h. Đủ để bắt lỗi vượt trần 4B.
    h, L, V = cfg.hidden_size, cfg.num_hidden_layers, cfg.vocab_size
    print(f"{m:<52} h={h} L={L} vocab={V:,}  ~{(12*L*h*h + V*h)/1e6:.0f}M")

import os
os.environ["E5"] = PATHS["intfloat/multilingual-e5-small"]
os.environ["BKAI"] = PATHS["bkai-foundation-models/vietnamese-bi-encoder"]


## 5 — Encode 287.358 chunk + đo trần

Encode một lần, cache `.npy` (287k × 384 fp16 = 0,22 GB). Không dùng thư viện vector
database: ở quy mô này matmul thẳng nhanh hơn, và HNSW là **xấp xỉ** — đổi độ chính xác
lấy tốc độ mà ta không cần.

Bảng in ra ba dòng BM25 / dense / hybrid. Dòng `dense` một mình cũng đáng nhìn: nếu nó
tệ hẳn thì RRF cũng không cứu, mà nếu nó ngang BM25 thì trộn mới có nghĩa.

In [ ]:
!python scripts/eval_hybrid.py --dense-model $E5 --n-cau 1000 --deep 1000 --batch-size 256

## 6 — Model thứ hai (PhoBERT-base, train riêng tiếng Việt)

In [ ]:
!python scripts/eval_hybrid.py --dense-model $BKAI --n-cau 1000 --deep 1000 --batch-size 256

## 7 — Nếu hybrid vượt ngưỡng: kiểm lại bằng cross-encoder thật

Chỉ chạy khi cell 5 hoặc 6 cho trần ≥ 98%. Ước "+0,7 điểm R@5 mỗi điểm trần" là suy ra
từ §15.6, không phải đo — cell này đo thẳng.

In [ ]:
# Đặt DENSE = $E5 hoặc $BKAI (cái thắng ở trên) rồi bỏ comment.
# !python scripts/eval_rerank.py --model $MODEL --pool 50 --n-cau 500 --batch-size 32